# Stage 2 metric foundation and synthetic validation

This notebook contains the same metric core used by the decoder and comparison notebooks, plus deterministic synthetic checks for empty/perfect masks, spacing, missing bones, fragments, gaps, bridges, and subject-level aggregation.

In [ ]:
import math
import numpy as np
import pandas as pd
import torch

In [ ]:
from scipy.ndimage import binary_erosion, distance_transform_edt, label

BONES = ["femur", "tibia", "patella", "fibula"]


def overlap_metrics(prediction, target):
    """Explicit empty handling: an empty target is invalid; an empty prediction against a target scores zero."""
    prediction = np.asarray(prediction, dtype=bool); target = np.asarray(target, dtype=bool)
    invalid_target = not target.any(); empty_prediction = not prediction.any()
    if invalid_target:
        return {"dice": float("nan"), "iou": float("nan"), "invalid_target": True, "empty_prediction": empty_prediction}
    if empty_prediction:
        return {"dice": 0.0, "iou": 0.0, "invalid_target": False, "empty_prediction": True}
    intersection = np.logical_and(prediction, target).sum(dtype=np.float64)
    pred_count = prediction.sum(dtype=np.float64); target_count = target.sum(dtype=np.float64)
    return {"dice": float(2 * intersection / (pred_count + target_count)), "iou": float(intersection / (pred_count + target_count - intersection)), "invalid_target": False, "empty_prediction": False}


def _surface_distances(prediction, target, spacing_xyz):
    prediction = np.asarray(prediction, dtype=bool); target = np.asarray(target, dtype=bool)
    spacing_xyz = tuple(float(value) for value in spacing_xyz)
    if len(spacing_xyz) != 3 or any(value <= 0 for value in spacing_xyz): raise ValueError("spacing_xyz must contain three positive millimetre values")
    pred_surface = prediction & ~binary_erosion(prediction); target_surface = target & ~binary_erosion(target)
    if not pred_surface.any() or not target_surface.any(): return None
    to_target = distance_transform_edt(~target_surface, sampling=spacing_xyz)[pred_surface]
    to_prediction = distance_transform_edt(~pred_surface, sampling=spacing_xyz)[target_surface]
    return np.concatenate([to_target, to_prediction])


def hd95_mm(prediction, target, spacing_xyz):
    distances = _surface_distances(prediction, target, spacing_xyz)
    return float("nan") if distances is None else float(np.percentile(distances, 95))


def assd_mm(prediction, target, spacing_xyz):
    distances = _surface_distances(prediction, target, spacing_xyz)
    return float("nan") if distances is None else float(distances.mean())


def per_bone_metrics(logits, target, spacing_xyz=(0.78125, 0.78125, 0.78125), threshold=0.5):
    probability = torch.sigmoid(logits.float()).detach().cpu().numpy(); truth = target.detach().cpu().numpy() > 0.5
    prediction = probability > threshold; rows = []
    for batch_index in range(prediction.shape[0]):
        record = {}
        for bone_index, bone in enumerate(BONES):
            pred_mask = prediction[batch_index, bone_index]; target_mask = truth[batch_index, bone_index]
            overlap = overlap_metrics(pred_mask, target_mask)
            record.update({f"dice_{bone}": overlap["dice"], f"iou_{bone}": overlap["iou"], f"hd95_mm_{bone}": hd95_mm(pred_mask, target_mask, spacing_xyz), f"assd_mm_{bone}": assd_mm(pred_mask, target_mask, spacing_xyz), f"empty_prediction_{bone}": overlap["empty_prediction"], f"invalid_target_{bone}": overlap["invalid_target"]})
        valid_dice = [record[f"dice_{bone}"] for bone in BONES if np.isfinite(record[f"dice_{bone}"])]
        valid_iou = [record[f"iou_{bone}"] for bone in BONES if np.isfinite(record[f"iou_{bone}"])]
        record["dice_macro"] = float(np.mean(valid_dice)) if valid_dice else float("nan"); record["iou_macro"] = float(np.mean(valid_iou)) if valid_iou else float("nan")
        rows.append(record)
    return rows


def two_largest_components(mask):
    labels, count = label(np.asarray(mask, dtype=bool))
    if count < 2: return None
    sizes = [(component, int((labels == component).sum())) for component in range(1, count + 1)]
    selected = sorted(sizes, key=lambda item: item[1], reverse=True)[:2]
    return labels == selected[0][0], labels == selected[1][0]


def minimum_component_gap_mm(mask, spacing_xyz=(0.78125, 0.78125, 0.78125)):
    pair = two_largest_components(mask)
    if pair is None: return float("nan")
    first, second = pair
    return float(distance_transform_edt(~second, sampling=spacing_xyz)[first].min())


def component_bridge_metrics(prediction, target, spacing_xyz=(0.78125, 0.78125, 0.78125)):
    _, pred_components = label(np.asarray(prediction, dtype=bool)); _, target_components = label(np.asarray(target, dtype=bool))
    return {"prediction_components": int(pred_components), "target_components": int(target_components), "component_agreement": bool(pred_components == target_components), "false_bridge": bool(target_components >= 2 and pred_components < target_components), "prediction_min_gap_mm": minimum_component_gap_mm(prediction, spacing_xyz), "target_min_gap_mm": minimum_component_gap_mm(target, spacing_xyz)}


def aggregate_subject_level(frame, metric_columns, subject_column="subject_id"):
    """Average knees within subject first, then average subjects so bilateral knees do not receive extra weight."""
    subject = frame.groupby(subject_column, as_index=False)[metric_columns].mean(numeric_only=True)
    return subject, subject[metric_columns].mean(numeric_only=True).to_dict()

In [ ]:
RESULTS=[]
def check(name,condition,detail=""):
    RESULTS.append((name,bool(condition),detail)); print("PASS" if condition else "FAIL","-",name,detail)
def one_voxel(index,shape=(40,40,40)):
    mask=np.zeros(shape,bool);mask[index]=True;return mask
def logits_from_masks(masks):
    tensor=torch.as_tensor(np.stack(masks),dtype=torch.bool)
    return torch.where(tensor,torch.tensor(8.0),torch.tensor(-8.0)).unsqueeze(0)

In [ ]:
target=np.zeros((32,32,32),bool);target[10:15,10:15,10:15]=True;empty=np.zeros_like(target)
empty_metrics=overlap_metrics(empty,target)
check("empty prediction Dice is zero",empty_metrics["dice"]==0.0)
check("empty prediction IoU is zero",empty_metrics["iou"]==0.0)
check("empty prediction flag",empty_metrics["empty_prediction"])
check("empty prediction HD95 is NaN",math.isnan(hd95_mm(empty,target,(0.78125,)*3)))
check("empty prediction ASSD is NaN",math.isnan(assd_mm(empty,target,(0.78125,)*3)))
invalid=overlap_metrics(empty,empty)
check("empty target is explicit invalid target",invalid["invalid_target"] and math.isnan(invalid["dice"]))
perfect=overlap_metrics(target,target)
check("perfect Dice",perfect["dice"]==1.0)
check("perfect IoU",perfect["iou"]==1.0)
check("perfect HD95",hd95_mm(target,target,(0.78125,)*3)==0.0)

In [ ]:
origin=one_voxel((20,20,20))
for offset in (1,3,7):
    shifted=one_voxel((20+offset,20,20));spacing=(0.7,0.7,0.7);expected=offset*0.7
    check(f"offset {offset} ASSD mm",abs(assd_mm(shifted,origin,spacing)-expected)<1e-9)
    check(f"offset {offset} HD95 mm",abs(hd95_mm(shifted,origin,spacing)-expected)<1e-9)
diagonal=one_voxel((23,20,23));expected=math.sqrt((3*3.0)**2+(3*0.7)**2)
check("anisotropic sampling-aware ASSD",abs(assd_mm(diagonal,origin,(3.0,0.7,0.7))-expected)<1e-9)

In [ ]:
fracture=np.zeros((40,40,40),bool);fracture[15:20,18:22,18:22]=True;fracture[24:29,18:22,18:22]=True
faithful=fracture.copy();bridged=fracture.copy();bridged[20:24,18:22,18:22]=True
faithful_metrics=component_bridge_metrics(faithful,fracture,(1.0,1.0,1.0));bridge_metrics=component_bridge_metrics(bridged,fracture,(1.0,1.0,1.0))
check("fracture has two components",faithful_metrics["target_components"]==2)
check("faithful component agreement",faithful_metrics["component_agreement"])
check("bridge collapses components",bridge_metrics["prediction_components"]==1)
check("false bridge flag",bridge_metrics["false_bridge"])
check("known fragment gap",abs(faithful_metrics["target_min_gap_mm"]-5.0)<1e-9,faithful_metrics["target_min_gap_mm"])

In [ ]:
frame=pd.DataFrame([{"subject_id":"s1","side":"L","dice_femur":0.9},{"subject_id":"s1","side":"R","dice_femur":0.7},{"subject_id":"s2","side":"L","dice_femur":0.5}])
subject,overall=aggregate_subject_level(frame,["dice_femur"])
check("bilateral knees aggregate within subject",abs(subject.loc[subject.subject_id.eq("s1"),"dice_femur"].iloc[0]-0.8)<1e-12)
check("subjects receive equal weight",abs(overall["dice_femur"]-0.65)<1e-12)

target4=np.stack([target]*4);prediction4=np.stack([target,target,target,empty]);rows=per_bone_metrics(logits_from_masks(list(prediction4)),torch.from_numpy(target4).unsqueeze(0).float())
check("per-bone output includes all ordered bones",all(f"dice_{bone}" in rows[0] for bone in BONES))
check("missing fibula cannot hide in macro",rows[0]["dice_fibula"]==0.0 and rows[0]["dice_macro"]<1.0)

In [ ]:
failed=[name for name,passed,_ in RESULTS if not passed]
print(f"=== {len(RESULTS)-len(failed)}/{len(RESULTS)} checks passed ===")
assert not failed,f"metric foundation failures: {failed}"